# Section 8 — Fine-Tuning & Open-Source Models

> *Knowing when to fine-tune — and when not to — is most of the skill.*

This notebook is the hands-on companion to Section 8. We run **two full tracks**:

**Track A — OpenAI Fine-Tuning** (Chapters 1–6)
- Prepare data in JSONL
- Upload, kick off a training job, monitor
- Call your fine-tuned model
- Bake-off against a prompt baseline

**Track B — LoRA on Open-Source** (Chapters 7–11)
- Load Llama 3.2 3B in 4-bit on a free Colab T4
- Attach LoRA adapters with PEFT
- Train on the same support-ticket task
- Three-way bake-off: prompt vs OpenAI FT vs LoRA

The goal isn't just to make the code run — it's to leave you with a working **decision framework** for the next time you face this choice.

## 0 — Setup

Track A (OpenAI FT) runs anywhere — just an API key.
Track B (LoRA) needs a GPU. The instructions target **free Colab T4**, but any 16 GB+ GPU works.

### 0.1 — Install dependencies

In [ ]:
# Track A: OpenAI fine-tuning
!pip install -q openai python-dotenv tiktoken tqdm

In [ ]:
# Track B: LoRA on open-source (skip if you're only running Track A)
# Pinned versions — bitsandbytes + peft + transformers drift quickly
!pip install -q transformers==4.44.2 peft==0.13.0 bitsandbytes==0.43.3 datasets==2.21.0 accelerate==0.34.2

### 0.2 — API keys

For Track A you need an OpenAI API key.
For Track B (Llama) you need a Hugging Face token. Visit `huggingface.co/meta-llama/Llama-3.2-3B-Instruct` and accept the terms first.

In [4]:
import os
from dotenv import load_dotenv
load_dotenv()

# Option 1: load from a .env file in the same directory
#   OPENAI_API_KEY=sk-...
#   HF_TOKEN=hf_...

# Option 2: paste directly (don't commit!)
# os.environ["OPENAI_API_KEY"] = "sk-..."
# os.environ["HF_TOKEN"] = "hf_..."

# Option 3 (Colab): use the Secrets tab (🔑 icon on the left sidebar)
# from google.colab import userdata
# os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
# os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")

assert os.environ.get("OPENAI_API_KEY"), "Set OPENAI_API_KEY for Track A"
print("✅ OpenAI key loaded")
if os.environ.get("HF_TOKEN"):
    print("✅ HF token loaded (Track B ready)")
else:
    print("ℹ️  HF token not set — Track B will fail when loading Llama")

✅ OpenAI key loaded
ℹ️  HF token not set — Track B will fail when loading Llama


### 0.3 — Helper functions

The usual suspects from prior sections.

In [5]:
from openai import OpenAI
client = OpenAI()

def ask(prompt, model="gpt-4o-mini", system=None, **kwargs):
    """Plain text completion."""
    messages = []
    if system:
        messages.append({"role": "system", "content": system})
    messages.append({"role": "user", "content": prompt})
    response = client.chat.completions.create(
        model=model, messages=messages, temperature=0, **kwargs
    )
    return response.choices[0].message.content

# Quick sanity check
print(ask("Reply with exactly the word PONG."))

PONG


---

## Chapter 1 — The Mental Model Check

Before we touch any code, a 60-second self-check. Fine-tuning teaches **style and behavior**, not facts. The most common failure mode is reaching for fine-tuning when the real problem is something else.

### 1.1 — The three forks

Most "I need fine-tuning" intuitions are actually one of these:

| Symptom | Better tool |
|---|---|
| "Model doesn't know my product/policy/data" | **RAG** — inject context at query time |
| "Model isn't smart enough" | **Bigger base model or better prompt** |
| "Model output style is inconsistent" | **Fine-tuning** ← genuinely the right tool |
| "We make 100k calls/day and bills hurt" | **Fine-tuning** ← distill into a cheaper model |
| "Need classification at scale" | **Fine-tuning** ← classic use case |

Our running example below — classifying support tickets into 5 categories — is the classic case for fine-tuning. We have a stable taxonomy, a repetitive task, and (we'll pretend) production volume that justifies the work.

---

## Chapter 2 — The Dataset

We'll build a realistic dataset: ~225 customer-support tickets across five categories.

### 2.1 — Why synthetic?

For teaching purposes we generate the data inline. In real projects you'd start from logs of actual tickets, hand-labeled by your support team. The principles are identical — only the source changes.

In [1]:
import random
random.seed(42)

TEMPLATES = {
    "Hardware": [
        "My laptop screen is flickering after the latest update",
        "Mouse stopped working completely this morning",
        "Keyboard keys are sticking, especially the space bar",
        "Monitor displays a black screen but PC is on",
        "Webcam not detected by any app",
        "Headphones jack seems broken — only one side works",
        "Printer keeps jamming on the third page",
        "Fan running constantly and very loud",
        "Battery drains in under an hour",
        "USB ports not recognizing any device",
        "Touchpad is unresponsive in random spots",
        "External hard drive isn't being detected",
        "Screen has dead pixels in the corner",
        "Laptop overheating and shutting down during meetings",
        "Power button doesn't turn the machine on",
    ],
    "Software": [
        "Excel keeps crashing when I open large files",
        "Outlook won't sync emails since this morning",
        "Word document corrupted — can I recover it?",
        "Adobe Reader gives an error opening PDFs",
        "Teams says I'm offline when I'm clearly online",
        "Application freezes whenever I try to print",
        "Browser keeps closing tabs without warning",
        "VPN client refuses to connect",
        "Update installer fails halfway through",
        "Antivirus is blocking a legitimate tool I need",
        "PowerPoint won't open .pptx attachments anymore",
        "Slack notifications stopped showing on desktop",
        "Software keeps reverting to an old version",
        "Cannot install the new accounting package",
        "App crashes on startup after Windows update",
    ],
    "Network": [
        "WiFi keeps disconnecting every 10 minutes",
        "Internet is extremely slow in the office today",
        "Can't reach internal sites from my home network",
        "VPN drops every time I switch between apps",
        "Network printer not reachable from my floor",
        "Ethernet port shows 'no connection' constantly",
        "Office WiFi requires re-login every hour",
        "DNS errors when visiting any external site",
        "Network drive disconnects randomly",
        "Slow file transfer to the shared server",
        "WiFi signal weak in the corner conference room",
        "Can't connect to guest network for visitors",
        "Internet works on phone but not laptop on same WiFi",
        "Network keeps dropping during video calls",
        "Firewall blocking access to a vendor portal",
    ],
    "Account": [
        "Can't log into my email account",
        "Password reset email never arrives",
        "Two-factor authentication codes aren't working",
        "Account is locked after one wrong attempt",
        "SSO redirects to a broken page",
        "My username isn't recognized anymore",
        "Need to update my registered phone number",
        "Permissions seem to have been removed overnight",
        "Can't access the shared drive folder I had before",
        "Account shows as deactivated by mistake",
        "Group membership appears to have changed",
        "Cannot log into the HR portal — credentials rejected",
        "Need to merge two accounts under the same email",
        "Profile picture won't update in any app",
        "Session expires every five minutes",
    ],
    "Billing": [
        "Wrong charge appeared on my last statement",
        "Refund request for order #4521",
        "Invoice attachment is missing from the email",
        "Need a copy of last year's annual invoice",
        "Charged twice for the same subscription",
        "Promo code didn't apply at checkout",
        "Tax calculation looks incorrect on my invoice",
        "Payment method was declined for no reason",
        "Need to update billing address on file",
        "Auto-renewal happened despite cancellation",
        "Currency conversion rate seems off",
        "Invoice number doesn't match my purchase order",
        "Refund processed but never appeared in account",
        "Need an itemized receipt for an expense report",
        "Subscription tier shows as different than what I bought",
    ],
}

data = []
for category, templates in TEMPLATES.items():
    for ticket in templates:
        data.append({"ticket": ticket, "category": category})
    # 2 variations per template → ~45 per class
    for ticket in templates:
        for v in ["Hi team, " + ticket.lower(), ticket + " — please help ASAP"]:
            data.append({"ticket": v, "category": category})

random.shuffle(data)

from collections import Counter
print(f"Total examples: {len(data)}\n")
for cat, n in Counter(d["category"] for d in data).items():
    print(f"  {cat:12s} {n:3d}")
print("\nSample 3:")
for d in data[:3]:
    print(f"  [{d['category']:8s}] {d['ticket']}")

Total examples: 225

  Network       45
  Billing       45
  Hardware      45
  Account       45
  Software      45

Sample 3:
  [Network ] Network drive disconnects randomly — please help ASAP
  [Billing ] Hi team, wrong charge appeared on my last statement
  [Hardware] Keyboard keys are sticking, especially the space bar


### 2.2 — Split into train / validation / test

The golden rule: split **before** doing anything else, and never let the test set leak into training.

In [2]:
from collections import defaultdict

by_class = defaultdict(list)
for d in data:
    by_class[d["category"]].append(d)

train_data, val_data, test_data = [], [], []
for cat, items in by_class.items():
    random.shuffle(items)
    n = len(items)
    train_data.extend(items[:int(0.6 * n)])
    val_data.extend(items[int(0.6 * n):int(0.8 * n)])
    test_data.extend(items[int(0.8 * n):])

random.shuffle(train_data)
random.shuffle(val_data)
random.shuffle(test_data)

print(f"Train:      {len(train_data):3d} examples")
print(f"Validation: {len(val_data):3d} examples")
print(f"Test:       {len(test_data):3d} examples (NEVER used during training)")

Train:      135 examples
Validation:  45 examples
Test:        45 examples (NEVER used during training)


💡 **Key takeaway:** Splits get done once, frozen, and never touched until evaluation. The test set is sacred — any peek at it during training contaminates your results.

---

## Chapter 3 — The Prompt-Engineered Baseline

**Always build this first.** If a careful prompt with few-shot examples matches the fine-tune, you saved yourself a week of work.

### 3.1 — The baseline prompt

In [6]:
BASELINE_SYSTEM = """You are a customer-support ticket classifier.

Categories: Hardware, Software, Network, Account, Billing

Examples:
- "My laptop screen flickers constantly" → Hardware
- "Outlook won't sync emails" → Software
- "WiFi disconnects every 10 minutes" → Network
- "Can't log into my account" → Account
- "Refund request for order #1234" → Billing
- "Mouse stopped working" → Hardware
- "Excel crashes on large files" → Software
- "VPN keeps dropping" → Network
- "2FA codes not working" → Account
- "Charged twice for subscription" → Billing

Respond with only the category name. No explanation."""

def predict_baseline(ticket):
    return ask(ticket, system=BASELINE_SYSTEM, model="gpt-4o-mini").strip()

# Quick check
print(predict_baseline("My screen is dim and flickering"))
print(predict_baseline("Need a refund for last month's bill"))
print(predict_baseline("Can't reach the internal wiki"))

Hardware
Billing
Network


### 3.2 — Evaluate the baseline on the test set

This is the number we have to beat.

In [7]:
from tqdm import tqdm

def evaluate(predict_fn, test_set, label="Model"):
    correct = 0
    errors = []
    for example in tqdm(test_set, desc=label):
        pred = predict_fn(example["ticket"]).strip()
        if pred == example["category"]:
            correct += 1
        else:
            errors.append({"ticket": example["ticket"], "true": example["category"], "pred": pred})
    accuracy = correct / len(test_set)
    print(f"\n{label} accuracy: {correct}/{len(test_set)} = {accuracy:.1%}")
    return accuracy, errors

baseline_acc, baseline_errors = evaluate(predict_baseline, test_data, label="Baseline (prompt + few-shot)")

Baseline (prompt + few-shot): 100%|██████████| 45/45 [00:28<00:00,  1.61it/s]


Baseline (prompt + few-shot) accuracy: 42/45 = 93.3%


In [8]:
# Patterns in the errors matter more than the count
print(f"Errors: {len(baseline_errors)}\n")
for e in baseline_errors[:8]:
    print(f"  [{e['true']:8s} → {e['pred']:8s}]  {e['ticket']}")

Errors: 3

  [Software → Network ]  VPN client refuses to connect
  [Software → Network ]  VPN client refuses to connect — please help ASAP
  [Account  → Network ]  Can't access the shared drive folder I had before


💡 **Key takeaway:** Look at errors qualitatively before deciding to fine-tune. If errors look like genuine ambiguity (a ticket that could plausibly be Account *or* Billing), fine-tuning won't help much — the data itself is ambiguous. If errors look like the model isn't picking up subtle patterns, fine-tuning probably will help.

---

# Track A — OpenAI Fine-Tuning

We now do the actual fine-tune. This runs on OpenAI's infra; no GPU needed locally.

Total wall-clock time: ~15–30 minutes (most of it is OpenAI's training queue).

## Chapter 4 — Preparing the JSONL

OpenAI expects a specific format: **one JSON object per line**, each containing a full chat conversation (system + user + assistant).

### 4.1 — Format your training data

In [9]:
import json

# Note: the FT system prompt is much shorter than the baseline,
# because the categories will be "baked into" the weights
FT_SYSTEM = "Classify the support ticket. Respond with one of: Hardware, Software, Network, Account, Billing."

def to_jsonl_line(example):
    return {
        "messages": [
            {"role": "system", "content": FT_SYSTEM},
            {"role": "user", "content": example["ticket"]},
            {"role": "assistant", "content": example["category"]},
        ]
    }

with open("train.jsonl", "w") as f:
    for ex in train_data:
        f.write(json.dumps(to_jsonl_line(ex)) + "\n")

with open("val.jsonl", "w") as f:
    for ex in val_data:
        f.write(json.dumps(to_jsonl_line(ex)) + "\n")

print("First training line:")
with open("train.jsonl") as f:
    print(f.readline().strip())
print(f"\nTotal: {len(train_data)} train, {len(val_data)} val")

First training line:
{"messages": [{"role": "system", "content": "Classify the support ticket. Respond with one of: Hardware, Software, Network, Account, Billing."}, {"role": "user", "content": "Payment method was declined for no reason"}, {"role": "assistant", "content": "Billing"}]}

Total: 135 train, 45 val


### 4.2 — Validate before uploading

OpenAI rejects malformed files. Catch problems locally first.

In [10]:
def validate_jsonl(path):
    problems = []
    with open(path) as f:
        for i, line in enumerate(f, 1):
            try:
                obj = json.loads(line)
                assert "messages" in obj, "no 'messages' key"
                roles = [m["role"] for m in obj["messages"]]
                assert "user" in roles and "assistant" in roles, "need user + assistant"
            except (json.JSONDecodeError, AssertionError, KeyError) as e:
                problems.append((i, str(e)))
    if problems:
        print(f"❌ {len(problems)} problems in {path}")
        for p in problems[:5]:
            print(f"  Line {p[0]}: {p[1]}")
    else:
        print(f"✅ {path} valid")

validate_jsonl("train.jsonl")
validate_jsonl("val.jsonl")

✅ train.jsonl valid
✅ val.jsonl valid


💡 **Key takeaway:** Catch JSONL formatting errors locally. OpenAI's error messages are terse and you'll burn time round-tripping if you upload a broken file.

---

## Chapter 5 — Running the Fine-Tuning Job

Three API calls: upload, start, poll.

### 5.1 — Upload

In [11]:
train_file = client.files.create(
    file=open("train.jsonl", "rb"),
    purpose="fine-tune"
)
print(f"Training file ID: {train_file.id}")

val_file = client.files.create(
    file=open("val.jsonl", "rb"),
    purpose="fine-tune"
)
print(f"Validation file ID: {val_file.id}")

Training file ID: file-STeA9jyPNZwsxio9CpXUvS
Validation file ID: file-FhLgjahVZBv81uxn3hYCB3


### 5.2 — Start the job

We'll fine-tune `gpt-4o-mini-2024-07-18`. Pinning the version matters — base models get deprecated and your fine-tune goes with them. Hyperparameters left on `auto` — OpenAI picks reasonable defaults for small datasets.

In [ ]:
job = client.fine_tuning.jobs.create(
    training_file=train_file.id,
    validation_file=val_file.id,
    model="gpt-4o-mini-2024-07-18",
    suffix="tickets-v1",  # appears in your model name for easy identification
)
print(f"Job started: {job.id}")
print(f"Status: {job.status}")

### 5.3 — Poll until done

A small job (~135 examples × 3 epochs by default) finishes in **10–20 minutes**. Larger jobs can take hours. This cell blocks until completion or failure.

In [ ]:
import time

start = time.time()
while True:
    job = client.fine_tuning.jobs.retrieve(job.id)
    elapsed = int(time.time() - start)
    print(f"[{elapsed:4d}s] Status: {job.status}")
    if job.status in ("succeeded", "failed", "cancelled"):
        break
    time.sleep(30)

print(f"\n--- Final status: {job.status} ---")
if job.status == "succeeded":
    print(f"Model ID: {job.fine_tuned_model}")
    print(f"Trained tokens: {job.trained_tokens:,}")
else:
    print(f"Error: {job.error}")

### 5.4 — Inspect what happened

OpenAI logs training and validation loss at each step. Sharp drop = learning. Plateau too early = needs more data or epochs. Validation loss rising while training loss falls = overfitting.

In [ ]:
events = client.fine_tuning.jobs.list_events(fine_tuning_job_id=job.id, limit=50)
for e in reversed(list(events)):
    if hasattr(e, "data") and e.data and isinstance(e.data, dict) and "step" in e.data:
        d = e.data
        val_loss = d.get('valid_loss', 'n/a')
        val_str = f"{val_loss:.4f}" if isinstance(val_loss, float) else str(val_loss)
        print(f"  step {d.get('step'):4d}  train_loss={d.get('train_loss'):.4f}  val_loss={val_str}")

💡 **Key takeaway:** Watch validation loss, not training loss. Training loss always goes down — that's just memorization. Validation loss is the one that tells you whether the model is actually learning the task.

---

## Chapter 6 — Evaluating the OpenAI Fine-Tune

The honest test: how does the fine-tuned model do on the held-out test set vs the baseline?

In [ ]:
FT_MODEL = job.fine_tuned_model
print(f"Using model: {FT_MODEL}")

def predict_ft_openai(ticket):
    return ask(ticket, system=FT_SYSTEM, model=FT_MODEL).strip()

ft_acc, ft_errors = evaluate(predict_ft_openai, test_data, label="OpenAI fine-tune")

### 6.1 — Side-by-side comparison

In [ ]:
print(f"\n{'='*50}")
print(f"  Baseline (prompt + few-shot):  {baseline_acc:.1%}")
print(f"  OpenAI fine-tune:              {ft_acc:.1%}")
print(f"  Delta:                         {(ft_acc - baseline_acc)*100:+.1f}pp")
print(f"{'='*50}\n")

# Token cost comparison
import tiktoken
enc = tiktoken.encoding_for_model("gpt-4o-mini")
baseline_in = sum(len(enc.encode(BASELINE_SYSTEM + d["ticket"])) for d in test_data) / len(test_data)
ft_in = sum(len(enc.encode(FT_SYSTEM + d["ticket"])) for d in test_data) / len(test_data)
print(f"Avg input tokens — baseline: {baseline_in:.0f},  fine-tune: {ft_in:.0f}")
print(f"Token reduction: {(1 - ft_in/baseline_in)*100:.0f}%")

### 6.2 — Inspect the wins and losses

Where did fine-tuning specifically help? Where did it hurt?

In [ ]:
baseline_wrong = {(e["ticket"], e["true"]) for e in baseline_errors}
ft_wrong = {(e["ticket"], e["true"]) for e in ft_errors}

ft_fixed = baseline_wrong - ft_wrong         # baseline got wrong, FT got right
ft_broke = ft_wrong - baseline_wrong         # FT got wrong, baseline got right
both_wrong = baseline_wrong & ft_wrong       # both got wrong

print(f"Fine-tune FIXED {len(ft_fixed)} cases the baseline missed")
for t, true in list(ft_fixed)[:5]:
    print(f"  [{true}] {t}")

print(f"\nFine-tune BROKE {len(ft_broke)} cases the baseline got right")
for t, true in list(ft_broke)[:5]:
    print(f"  [{true}] {t}")

print(f"\nBoth wrong on {len(both_wrong)} cases (genuine ambiguity)")

💡 **Key takeaway:** A fine-tune that gains 2pp average accuracy while breaking cases that used to work isn't an unambiguous win. Look at the actual cases — sometimes the baseline was right in a way that matters more to your users.

---

# Track B — LoRA on Open-Source Models

We now do the same task with an open-source model and LoRA. Runtime target: **free Colab T4** (16 GB GPU).

If you're on Colab, switch the runtime before continuing:
**Runtime → Change runtime type → T4 GPU**.

If you're local with a different GPU, the code works as-is on any 12 GB+ GPU.

## Chapter 7 — Loading Llama 3.2 in 4-bit

A 3B model needs ~6 GB at fp16. With 4-bit quantization (QLoRA's "Q") it fits in ~2 GB, leaving plenty of room for LoRA training overhead on a T4.

In [ ]:
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("⚠️  No GPU detected. Switch runtime to T4 GPU on Colab.")

In [ ]:
# Authenticate with Hugging Face (required for Llama)
from huggingface_hub import login
login(token=os.environ.get("HF_TOKEN"))
print("✅ Logged into Hugging Face")

### 7.1 — Load model and tokenizer in 4-bit

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

MODEL_ID = "meta-llama/Llama-3.2-3B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",                # NormalFloat4 — recommended by the QLoRA paper
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,           # quantize the quantization constants too
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token     # Llama doesn't ship with a pad token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
)
print(f"✅ Loaded {MODEL_ID}")
print(f"GPU memory used: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

### 7.2 — Sanity-check inference

Confirm the model runs before we try to train it.

In [ ]:
def generate(prompt, max_new_tokens=20):
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    outputs = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id,
    )
    return tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)

messages = [
    {"role": "system", "content": FT_SYSTEM},
    {"role": "user", "content": "WiFi keeps dropping every 10 minutes"},
]
prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
print(generate(prompt))

💡 **Key takeaway:** Even before training, the base instruct model handles classification reasonably — chat templates exist for a reason. The fine-tune's job is to make this faster, cheaper, and more consistent — not to teach the model classification from scratch.

---

## Chapter 8 — Attaching LoRA Adapters

Three steps: prepare the model for low-bit training, configure LoRA, wrap the model.

In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

# Step 1: handles low-bit training quirks (keeps layer norms in fp32, etc.)
model = prepare_model_for_kbit_training(model)

# Step 2: configure LoRA
lora_config = LoraConfig(
    r=16,                                # rank — how thin the trainable matrices are
    lora_alpha=32,                       # scaling factor (rule of thumb: 2 × r)
    target_modules=["q_proj", "v_proj"], # attach to attention query + value projections
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

# Step 3: wrap the base model — only LoRA params will train
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

Around **0.2% of parameters are actually trainable**. That's the LoRA magic — 99.8% of the model is frozen, but we still get meaningful behavior change from training the tiny adapter.

---

## Chapter 9 — Preparing Data for the Hugging Face Trainer

The Hugging Face `Trainer` wants tokenized examples. We use the same chat template the model was trained with.

In [ ]:
from datasets import Dataset

def format_for_training(example):
    """Build the full chat string the model should learn to produce."""
    messages = [
        {"role": "system", "content": FT_SYSTEM},
        {"role": "user", "content": example["ticket"]},
        {"role": "assistant", "content": example["category"]},
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False)
    return {"text": text}

train_ds = Dataset.from_list(train_data).map(format_for_training)
val_ds   = Dataset.from_list(val_data).map(format_for_training)

print("Example training text:\n")
print(train_ds[0]["text"][:500])
print("...")

### 9.1 — Tokenize

In [ ]:
def tokenize(example):
    out = tokenizer(
        example["text"],
        truncation=True,
        max_length=256,
        padding="max_length",
    )
    out["labels"] = out["input_ids"].copy()   # for causal LM, labels = inputs (shifted internally)
    return out

train_ds = train_ds.map(tokenize, remove_columns=["text", "ticket", "category"])
val_ds   = val_ds.map(tokenize, remove_columns=["text", "ticket", "category"])
print(f"Train: {len(train_ds)}  Val: {len(val_ds)}")

💡 **Key takeaway:** Causal LM training is "predict the next token at every position". The trainer doesn't need a separate "label" — it shifts the input by one internally. Setting `labels = input_ids` is the standard idiom.

---

## Chapter 10 — Running the LoRA Training

The actual training is anti-climactic — `Trainer` handles everything. Expect **15–25 minutes on a free Colab T4** for ~135 examples × 3 epochs.

In [ ]:
from transformers import Trainer, TrainingArguments

training_args = TrainingArguments(
    output_dir="./llama-tickets-lora",
    per_device_train_batch_size=4,
    gradient_accumulation_steps=2,       # effective batch size = 8
    num_train_epochs=3,
    learning_rate=2e-4,                  # LoRA tolerates higher LR than full FT
    fp16=True,
    logging_steps=10,
    save_strategy="epoch",
    eval_strategy="epoch",
    report_to="none",                    # disable W&B etc.
    remove_unused_columns=False,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    tokenizer=tokenizer,
)

trainer.train()

### 10.1 — Save the adapter

The output is a tiny file (~50–100 MB) — just the LoRA weights. The base model isn't copied; you load it separately at inference time.

In [ ]:
ADAPTER_DIR = "./llama-tickets-adapter-final"
model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)

import os
total = sum(
    os.path.getsize(os.path.join(ADAPTER_DIR, f))
    for f in os.listdir(ADAPTER_DIR)
    if os.path.isfile(os.path.join(ADAPTER_DIR, f))
)
print(f"Adapter size: {total / 1e6:.1f} MB")
print(f"(For comparison, the base Llama 3.2 3B is ~6 GB.)")

💡 **Key takeaway:** What you ship is a 50 MB adapter, not a 6 GB model. The same base model can host dozens of specialized adapters — you switch behaviors by loading a different adapter, not by reloading 6 GB of weights.

---

## Chapter 11 — Evaluating the LoRA Fine-Tune

In [ ]:
model.eval()

CATEGORIES = ["Hardware", "Software", "Network", "Account", "Billing"]

def predict_lora(ticket):
    messages = [
        {"role": "system", "content": FT_SYSTEM},
        {"role": "user", "content": ticket},
    ]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=10,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )
    response = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
    # Pull out just the category — model sometimes adds whitespace or extra tokens
    for cat in CATEGORIES:
        if cat in response:
            return cat
    return response.strip()

print(predict_lora("My laptop screen is flickering"))
print(predict_lora("Need a refund for last month"))

In [ ]:
lora_acc, lora_errors = evaluate(predict_lora, test_data, label="LoRA fine-tune (Llama 3.2 3B)")

---

## Chapter 12 — The Three-Way Bake-Off

The whole point of this section: line up all three approaches and compare them honestly.

In [ ]:
print(f"\n{'='*60}")
print(f"  {'Approach':<40s} {'Accuracy':>10s}")
print(f"  {'-'*40} {'-'*10}")
print(f"  {'Baseline (GPT-4o-mini + 10-shot prompt)':<40s} {baseline_acc:>9.1%}")
print(f"  {'OpenAI fine-tune (gpt-4o-mini)':<40s} {ft_acc:>9.1%}")
print(f"  {'LoRA fine-tune (Llama 3.2 3B)':<40s} {lora_acc:>9.1%}")
print(f"{'='*60}\n")

### 12.1 — Cost & latency comparison (rough)

Illustrative — exact numbers depend on your traffic. Orders of magnitude are what matter.

In [ ]:
# Per-1000-calls cost estimate
# Sources: OpenAI public pricing; LoRA assumes Together AI's Llama 3.2 3B serverless rate
print(f"\nApproximate cost per 1,000 classifications:")
print(f"  Baseline (GPT-4o + few-shot)       ~$0.85")
print(f"  Baseline (GPT-4o-mini + few-shot)  ~$0.06")
print(f"  OpenAI fine-tune (mini)            ~$0.02   ← prompt is ~1/10 the size")
print(f"  LoRA on Together / Fireworks (3B)  ~$0.005  ← smallest model, cheapest tokens")

print(f"\nLatency notes (rough, at single-call):")
print(f"  Baseline:   ~700-900ms (long prompt, mid-sized model)")
print(f"  OpenAI FT:  ~250-400ms (short prompt, mini model)")
print(f"  LoRA local: ~100-200ms (smallest model on consumer GPU)")

### 12.2 — Where did each approach win or lose?

In [ ]:
baseline_set = {e["ticket"] for e in baseline_errors}
ft_set       = {e["ticket"] for e in ft_errors}
lora_set     = {e["ticket"] for e in lora_errors}

print(f"Errors per approach:")
print(f"  Baseline:   {len(baseline_set)}")
print(f"  OpenAI FT:  {len(ft_set)}")
print(f"  LoRA:       {len(lora_set)}")
print()
print(f"Cases ONLY baseline got wrong:  {len(baseline_set - ft_set - lora_set)}")
print(f"Cases ONLY OpenAI FT got wrong: {len(ft_set - baseline_set - lora_set)}")
print(f"Cases ONLY LoRA got wrong:      {len(lora_set - baseline_set - ft_set)}")
print(f"Cases ALL THREE got wrong:      {len(baseline_set & ft_set & lora_set)}")

💡 **Key takeaway:** "Cases all three got wrong" is the most interesting bucket. Those examples are usually genuinely ambiguous in your training data — they reveal taxonomy problems, not model problems. Fixing those usually means clarifying your labeling guidelines, not more training.

---

## Chapter 13 — Your Decision Framework

Now write it down. Based on what you just measured:

### 13.1 — The questions to ask, in order

1. **Is the gap between baseline and fine-tune > 3pp?**
   - No → ship the baseline. Fewer moving parts, easier to update.
   - Yes → continue.

2. **Are the gains in cases that actually matter for users?**
   - Skim the "FT fixed" set qualitatively. Are these the cases you'd want a human to handle better?
   - If the fine-tune gains are on niche edge cases nobody really hits, the savings are illusory.

3. **What's the daily volume?**
   - Less than 1k/day → training cost dominates; baseline wins.
   - 1k–10k/day → break-even within weeks if accuracy gain is real.
   - 10k+/day → fine-tuning almost always wins on total cost.

4. **Privacy and control needs?**
   - Data must stay in your VPC → LoRA on open-source.
   - Need to pin behavior across model deprecations → LoRA gives you the artifact; OpenAI FT doesn't.

5. **Team operational capacity?**
   - Small team, ship fast → OpenAI FT (managed).
   - Platform team available → LoRA (cheaper at scale).

### 13.2 — Write your own version

Copy the template below into a new markdown cell. Fill it in with what you learned from your specific bake-off. Future-you will thank present-you.

```markdown
## My Fine-Tuning Decision Framework

**Task:** [describe]

**Volume:** [N calls/day]

**Approach I'd ship and why:**
- [ ] Baseline (prompt + few-shot)
- [ ] OpenAI fine-tune
- [ ] LoRA on open-source
- [ ] Hybrid (FT + RAG)

**Measured numbers:**
- Baseline accuracy: __%
- Fine-tune accuracy: __%
- Cost per 1k calls (baseline → FT): $___ → $___
- Latency (baseline → FT): __ms → __ms

**Reasons I'd revisit this decision:**
1. ...
2. ...
3. ...
```

---

## Wrap-Up

You've now done — end to end — both the OpenAI-managed and the open-source LoRA paths for the same problem. A few things worth pinning down:

**The thing that actually mattered:** the baseline. Most of the time you'll find a careful prompt is within a few points of the fine-tune. The fine-tune wins are usually on cost and latency at scale, not raw accuracy.

**The thing that surprises most people:** how small the LoRA adapter is. 50 MB to fundamentally change a 3B model's behavior. You can keep dozens of these around.

**The thing to never forget:** evaluation. A fine-tune that "feels better" but you didn't measure properly is a liability — you're committed to maintaining a model you don't actually know is better.

**Next section:** evaluation and observability. How do you know whether any of this is actually working in production?

### Exercises

1. **Rerun the baseline with 0 few-shot examples.** How far does accuracy drop? This shows how much of the baseline's quality came from the examples vs. the model itself.

2. **Increase LoRA rank from 16 → 32.** Does accuracy improve? At what cost (memory, training time)?

3. **Hold out one category from training.** Can the model still classify it? (Spoiler: poorly. This is why fine-tuning isn't a substitute for the training distribution covering your actual cases.)

4. **Mix in 20 deliberately ambiguous tickets** with random labels. Watch what it does to accuracy. This is why label quality > label quantity.

5. **Distill GPT-4o into a fine-tuned mini.** Run GPT-4o on 200 unlabeled tickets, use its predictions as training labels for mini, fine-tune. Compare to the human-labeled fine-tune. This is the most common production pattern.